# RTV Check-in Image Classifier — Full Pipeline

End-to-end ML solution that classifies Raising The Village field check-in images into 9 operational categories.

| Section | Task | Description |
|---------|------|-------------|
| 0 | Setup | Install exact project dependencies |
| 1 | Data Download | Fetch dataset zip from Google Drive |
| 2 | Task 1 — EDA | Exploratory data analysis with full statistical report |
| 3 | Task 1 — Pipeline | Stratified split, augmentation, class weights, YAML |
| 4 | Task 2 — Training | YOLO11 fine-tuning (single or two-phase) |
| 5 | Task 2 — Evaluation | Metrics, confusion matrix, F1 chart, error audit |
| 6 | Export | Save model + metrics to Google Drive |

> **Run in Google Colab.** Set **Runtime → Change runtime type → T4 GPU** for fast training.

---
## 0. Setup

Install the same dependencies used in the project's `requirements.txt`.

In [ ]:
# Core ML
!pip install -q torch>=2.0.0 torchvision>=0.15.0 ultralytics>=8.3.0

# Data science
!pip install -q numpy>=1.24.0 pandas>=2.0.0 scikit-learn>=1.3.0

# Visualisation
!pip install -q matplotlib>=3.7.0 seaborn>=0.12.0

# Image processing
!pip install -q Pillow>=10.0.0 opencv-python-headless>=4.8.0

# Utilities
!pip install -q tqdm>=4.65.0 gdown

In [ ]:
import os, sys, json, random, re, shutil, zipfile
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from PIL import Image, ImageEnhance, ImageFilter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 42
IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff'}
random.seed(SEED)
np.random.seed(SEED)

print(f"Python:  {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")

---
## 1. Download Dataset from Google Drive

**Option A** — Paste a Google Drive sharing URL:
  - Right-click your zip in Drive → Share → Copy link
  - Link format: `https://drive.google.com/file/d/XXXXXXX/view?usp=sharing`

**Option B** — Mount Drive and set the zip path directly.

In [ ]:
# >>> SET ONE OF THESE <<<
GDRIVE_URL = ""  # Paste sharing URL here

# OR uncomment these 3 lines:
# from google.colab import drive
# drive.mount('/content/drive')
# GDRIVE_ZIP_PATH = "/content/drive/MyDrive/path/to/data.zip"

In [ ]:
DATA_DIR = "data"
ZIP_PATH = "dataset.zip"

def extract_file_id(url):
    for pat in [r'/file/d/([a-zA-Z0-9_-]+)', r'id=([a-zA-Z0-9_-]+)', r'/d/([a-zA-Z0-9_-]+)']:
        m = re.search(pat, url)
        if m: return m.group(1)
    raise ValueError(f"Could not extract file ID from: {url}")

if GDRIVE_URL.strip():
    import gdown
    fid = extract_file_id(GDRIVE_URL)
    gdown.download(f"https://drive.google.com/uc?id={fid}&export=download",
                   ZIP_PATH, quiet=False, fuzzy=True)
    print(f"Downloaded: {os.path.getsize(ZIP_PATH)/1e6:.1f} MB")
elif 'GDRIVE_ZIP_PATH' in dir() and os.path.exists(GDRIVE_ZIP_PATH):
    shutil.copy2(GDRIVE_ZIP_PATH, ZIP_PATH)
    print(f"Copied from Drive: {os.path.getsize(ZIP_PATH)/1e6:.1f} MB")
else:
    print("ERROR: Set GDRIVE_URL or GDRIVE_ZIP_PATH above.")

In [ ]:
if os.path.exists(ZIP_PATH):
    if os.path.exists(DATA_DIR): shutil.rmtree(DATA_DIR)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf: zf.extractall('.')

    # Auto-detect root folder if zip wraps data in a parent dir
    if not os.path.exists(DATA_DIR):
        expected = {'compost', 'pigsty', 'tippytap', 'vsla', 'organic'}
        for item in os.listdir('.'):
            if os.path.isdir(item) and item not in (DATA_DIR, '__MACOSX', '.config'):
                if expected & set(os.listdir(item)):
                    os.rename(item, DATA_DIR)
                    print(f"Renamed '{item}' -> '{DATA_DIR}'")
                    break
    if os.path.exists('__MACOSX'): shutil.rmtree('__MACOSX')

    print("\nDataset categories:")
    for d in sorted(os.listdir(DATA_DIR)):
        full = os.path.join(DATA_DIR, d)
        if os.path.isdir(full) and not d.startswith('.'):
            print(f"  {d:25s}: {len(os.listdir(full))} files")
else:
    print("No zip found. Run the download cell above.")

---
## 2. Task 1 — Exploratory Data Analysis

Performs the same analysis as `task1_data_analysis.py`:
- Scan every image with `PIL.verify()` to detect corrupt files
- Collect metadata: dimensions, aspect ratio, file size, colour mode, megapixels
- Generate 5 plots: class distribution, dimension scatter, aspect ratio histogram, file size box plot, sample grid
- Print a full statistical summary with key observations

In [ ]:
# Scan dataset with corrupt image detection (matches task1_data_analysis.py)
records = []
corrupt_count = 0

for cat_dir in sorted(Path(DATA_DIR).iterdir()):
    if not cat_dir.is_dir() or cat_dir.name.startswith(('.', '_')):
        continue
    for img_file in sorted(cat_dir.iterdir()):
        if img_file.suffix.lower() not in IMG_EXTENSIONS:
            continue
        try:
            with Image.open(img_file) as img:
                img.verify()  # detect truncated / corrupt files
            with Image.open(img_file) as img:  # re-open after verify (PIL requirement)
                w, h = img.size
                records.append({
                    'category': cat_dir.name,
                    'filename': img_file.name,
                    'path': str(img_file),
                    'width': w,
                    'height': h,
                    'aspect_ratio': round(w / h, 3),
                    'channels': len(img.getbands()),
                    'color_mode': img.mode,
                    'file_size_kb': round(img_file.stat().st_size / 1024, 1),
                    'megapixels': round((w * h) / 1e6, 2),
                })
        except Exception as e:
            corrupt_count += 1
            print(f"[CORRUPT] {img_file}: {e}")

df = pd.DataFrame(records)
counts = df['category'].value_counts()
imbalance = counts.max() / counts.min()

print(f"Total valid images: {len(df)}")
print(f"Corrupt/skipped:    {corrupt_count}")
print(f"Categories:         {df['category'].nunique()}")
print(f"Imbalance ratio:    {imbalance:.1f}:1\n")
for cat in counts.index:
    print(f"  {cat:25s}: {counts[cat]:4d}  ({counts[cat]/len(df)*100:.1f}%)")

In [ ]:
# Plot 1: Class distribution (matches task1_data_analysis.plot_class_distribution)
sorted_counts = counts.sort_values()
fig, ax = plt.subplots(figsize=(10, 6))
palette = sns.color_palette('viridis', len(sorted_counts))
bars = ax.barh(sorted_counts.index, sorted_counts.values, color=palette)
for bar, val in zip(bars, sorted_counts.values):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=11, fontweight='bold')
ax.set_xlabel('Number of Images', fontsize=12)
ax.set_title(f'Class Distribution (Imbalance Ratio: {imbalance:.1f}:1)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Image dimensions scatter (matches task1_data_analysis.plot_image_dimensions)
categories = sorted(df['category'].unique())
fig, ax = plt.subplots(figsize=(10, 8))
palette = sns.color_palette('husl', len(categories))
for cat, color in zip(categories, palette):
    sub = df[df['category'] == cat]
    ax.scatter(sub['width'], sub['height'], label=cat, alpha=0.6, s=25, color=color)
ax.set_xlabel('Width (px)')
ax.set_ylabel('Height (px)')
ax.set_title('Image Dimensions by Category')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Aspect ratio histogram (matches task1_data_analysis.plot_aspect_ratio_histogram)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['aspect_ratio'], bins=40, color='steelblue', edgecolor='white')
ax.axvline(x=1.0, color='red', linestyle='--', label='Square (1:1)')
ax.set_xlabel('Aspect Ratio (W/H)')
ax.set_ylabel('Count')
ax.set_title('Aspect Ratio Distribution')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 4: File size box plot (matches task1_data_analysis.plot_file_size_distribution)
order = df.groupby('category')['file_size_kb'].median().sort_values().index
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='file_size_kb', y='category', order=order, ax=ax, palette='viridis')
ax.set_xlabel('File Size (KB)', fontsize=12)
ax.set_title('File Size Distribution by Category', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 5: Sample images grid (matches task1_data_analysis.plot_sample_grid)
n_samples = 4
fig, axes = plt.subplots(len(categories), n_samples, figsize=(n_samples*3, len(categories)*2.5))
for i, cat in enumerate(categories):
    cat_df = df[df['category'] == cat]
    sampled = cat_df.sample(n=min(n_samples, len(cat_df)), random_state=SEED)
    for j in range(n_samples):
        ax = axes[i][j]
        ax.set_xticks([]); ax.set_yticks([])
        if j < len(sampled):
            img = Image.open(sampled.iloc[j]['path']).convert('RGB')
            ax.imshow(img)
        if j == 0:
            ax.set_ylabel(cat, fontsize=9, rotation=0, labelpad=80, ha='right')
fig.suptitle('Sample Images per Category', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Full statistical summary (matches task1_data_analysis.generate_report)
print("=" * 65)
print("  DATASET ANALYSIS REPORT")
print("=" * 65)
print(f"Total images:       {len(df)}")
print(f"Corrupt/skipped:    {corrupt_count}")
print(f"Classes:            {df['category'].nunique()}")
print(f"Min class:          {counts.idxmin()} ({counts.min()})")
print(f"Max class:          {counts.idxmax()} ({counts.max()})")
print(f"Imbalance ratio:    {imbalance:.1f}:1")
print(f"\nWidth:   min={df['width'].min()}, max={df['width'].max()}, "
      f"mean={df['width'].mean():.0f}, std={df['width'].std():.0f}")
print(f"Height:  min={df['height'].min()}, max={df['height'].max()}, "
      f"mean={df['height'].mean():.0f}, std={df['height'].std():.0f}")
print(f"File KB: min={df['file_size_kb'].min()}, max={df['file_size_kb'].max()}, "
      f"mean={df['file_size_kb'].mean():.0f}")
print(f"Colour modes: {dict(df['color_mode'].value_counts())}")
print(f"\nKey observations:")
print(f"  1. Severe class imbalance — guinea-pig-shelter has ~{counts.min()} vs ~{counts.max()}")
print(f"  2. Visually similar categories (compost/organic/liquid-organic) may confuse model")
print(f"  3. Small dataset (~{len(df)} images) — transfer learning is non-negotiable")
print(f"  4. Mixed resolutions — standardise to 640x640 for YOLO")
print(f"  5. Field conditions: variable lighting, angles, backgrounds, partial views")
print("=" * 65)

---
## 3. Task 1 — Data Pipeline

Mirrors `task1_data_pipeline.py` — builds a reproducible pipeline:

**Design decisions:**
- **Stratified splitting** ensures proportional representation even for the 16-image minority class
- **Offline augmentation** (rather than only online) lets us visually inspect synthetic images and guarantees the minority class reaches a usable training volume before training begins
- **Target 80% of majority count** — full equalisation on 16 originals risks memorising augmentation artefacts
- **10 augmentation functions** matching the full script: flip, small/large rotation, brightness, contrast, saturation, blur, sharpness, random crop, combined multi-transform
- **Inverse-frequency class weights** saved as JSON for loss weighting
- **dataset.yaml** created for YOLO compatibility
- **Split before augmentation** to prevent data leakage

In [ ]:
OUTPUT_DIR = 'prepared_data'
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

paths  = df['path'].tolist()
labels = df['category'].tolist()

# Two-step stratified split (matches task1_data_pipeline.stratified_split)
train_val_p, test_p, train_val_l, test_l = train_test_split(
    paths, labels, test_size=TEST_RATIO, stratify=labels, random_state=SEED)
val_frac = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_p, val_p, train_l, val_l = train_test_split(
    train_val_p, train_val_l, test_size=val_frac, stratify=train_val_l, random_state=SEED)

splits = {
    'train': list(zip(train_p, train_l)),
    'val':   list(zip(val_p, val_l)),
    'test':  list(zip(test_p, test_l)),
}

print(f"Split ratios: {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}\n")
for name, data in splits.items():
    c = Counter(l for _, l in data)
    print(f"  {name:5s}: {len(data):4d} — {dict(sorted(c.items()))}")

In [ ]:
# Build YOLO directory structure (matches task1_data_pipeline.build_dataset)
if os.path.exists(OUTPUT_DIR): shutil.rmtree(OUTPUT_DIR)

for split_name, data in splits.items():
    for img_path, label in data:
        dest = os.path.join(OUTPUT_DIR, split_name, label)
        os.makedirs(dest, exist_ok=True)
        shutil.copy2(img_path, os.path.join(dest, os.path.basename(img_path)))

print("Copied originals into train/val/test.")

In [ ]:
# 10 augmentation functions (matches task1_data_pipeline exactly)
def _flip(img):     return img.transpose(Image.FLIP_LEFT_RIGHT)
def _rot_sm(img):   return img.rotate(random.uniform(-15, 15), fillcolor=(0,0,0))
def _rot_lg(img):   return img.rotate(random.uniform(-30, 30), fillcolor=(0,0,0))
def _bright(img):   return ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.3))
def _contrast(img): return ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.3))
def _color(img):    return ImageEnhance.Color(img).enhance(random.uniform(0.7, 1.3))
def _blur(img):     return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))
def _sharp(img):    return ImageEnhance.Sharpness(img).enhance(random.uniform(0.5, 2.0))
def _crop(img):
    """Random crop 80-100% of image and resize back."""
    w, h = img.size
    f = random.uniform(0.80, 1.0)
    nw, nh = int(w*f), int(h*f)
    left, top = random.randint(0, w-nw), random.randint(0, h-nh)
    return img.crop((left, top, left+nw, top+nh)).resize((w, h), Image.BILINEAR)
def _combined(img):
    """Compose several light transforms — mimics real field variation."""
    if random.random() > 0.5: img = _flip(img)
    img = img.rotate(random.uniform(-10, 10), fillcolor=(0,0,0))
    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.8, 1.2))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(0.8, 1.2))
    return img

AUG_FNS = [_flip, _rot_sm, _rot_lg, _bright, _contrast, _color, _blur, _sharp, _crop, _combined]

# Oversample minority classes to 80% of majority (matches compute_augmentation_plan)
train_counts = Counter(l for _, l in splits['train'])
target = int(max(train_counts.values()) * 0.80)

for label, count in train_counts.items():
    needed = target - count
    if needed <= 0: continue
    print(f"Augmenting {label}: +{needed} synthetic images (from {count} originals)")
    src_imgs = [p for p, l in splits['train'] if l == label]
    dest_dir = os.path.join(OUTPUT_DIR, 'train', label)
    for i in range(needed):
        src = random.choice(src_imgs)
        img = Image.open(src).convert('RGB')
        aug = AUG_FNS[i % len(AUG_FNS)](img)
        aug.save(os.path.join(dest_dir, f'aug_{i:04d}_{os.path.basename(src)}'), quality=95)

print("\nFinal training set:")
for d in sorted(Path(OUTPUT_DIR, 'train').iterdir()):
    if d.is_dir(): print(f"  {d.name:25s}: {len(list(d.glob('*')))}")

In [ ]:
# Class weights: w_c = N / (C * n_c) — (matches task1_data_pipeline.compute_class_weights)
tc = Counter(l for _, l in splits['train'])
total_train, n_cls = sum(tc.values()), len(tc)
class_weights = {lab: round(total_train / (n_cls * cnt), 4) for lab, cnt in tc.items()}

print("Class weights (inverse frequency):")
for lab in sorted(class_weights):
    print(f"  {lab:25s}: {class_weights[lab]:.3f}")

with open(os.path.join(OUTPUT_DIR, 'class_weights.json'), 'w') as f:
    json.dump(class_weights, f, indent=2)
print(f"\nSaved {OUTPUT_DIR}/class_weights.json")

In [ ]:
# Create dataset.yaml (matches task1_data_pipeline.write_dataset_yaml)
class_names = sorted(df['category'].unique())
yaml_lines = [
    f'# RTV Check-in Image Classification Dataset (seed={SEED})',
    f'path: {os.path.abspath(OUTPUT_DIR)}',
    'train: train', 'val: val', 'test: test', '',
    f'nc: {len(class_names)}', '', 'names:',
] + [f'  {i}: {n}' for i, n in enumerate(class_names)]

yaml_path = os.path.join(OUTPUT_DIR, 'dataset.yaml')
with open(yaml_path, 'w') as f: f.write('\n'.join(yaml_lines) + '\n')
print(f"Created {yaml_path}:")
print(open(yaml_path).read())

---
## 4. Task 2 — Model Training

Mirrors `task2_model_training.py` with the same design rationale:

**Model choice — YOLO11-cls:**
- Provides a strong ImageNet-pretrained backbone with a lightweight classification head
- Transfer learning is critical: ~1,200 images is far too few to train from scratch
- Built-in augmentation pipeline and cosine LR scheduling

**Two-phase strategy (recommended for small datasets):**
- Phase 1: Freeze backbone, train classifier head only (10 epochs, higher LR)
  - Rapidly adapts the output layer without disrupting pretrained features
- Phase 2: Unfreeze all layers, fine-tune end-to-end (remaining epochs, lower LR)
  - Backbone gradually adapts to the RTV domain

**Pitfalls addressed:**
- Overfitting → early stopping, cosine LR, weight decay, augmentation, mixup, random erasing
- Class imbalance → offline augmentation from pipeline + class-weighted loss
- Data leakage → split happens before augmentation; val/test never augmented

In [ ]:
from ultralytics import YOLO

MODEL_NAME = 'yolo11n-cls.pt'  # Options: yolo11n/s/m/l-cls.pt
EPOCHS     = 50
IMGSZ      = 640
BATCH      = 32
LR         = 0.001
PATIENCE   = 15
TWO_PHASE  = True   # Recommended for small datasets

DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print(f"Device:   {DEVICE}")
print(f"Model:    {MODEL_NAME}")
print(f"Epochs:   {EPOCHS} | Batch: {BATCH} | LR: {LR} | Patience: {PATIENCE}")
print(f"Strategy: {'Two-phase (freeze → finetune)' if TWO_PHASE else 'Single-phase'}")

In [ ]:
if TWO_PHASE:
    # Phase 1: classifier head only (matches train_two_phase)
    print("=" * 60)
    print("PHASE 1 — Classifier head only (backbone frozen, 10 epochs)")
    print("=" * 60)
    model = YOLO(MODEL_NAME)
    model.train(
        data=OUTPUT_DIR, epochs=10, imgsz=IMGSZ, batch=BATCH,
        lr0=0.01, lrf=0.1, patience=10, device=DEVICE,
        project='training', name='phase1_head', exist_ok=True,
        pretrained=True, optimizer='AdamW', freeze=10,
        warmup_epochs=2, cos_lr=True, seed=SEED, workers=2, verbose=True,
    )

    # Phase 2: full fine-tuning
    print("\n" + "=" * 60)
    print(f"PHASE 2 — Full fine-tuning ({EPOCHS-10} epochs)")
    print("=" * 60)
    model = YOLO('training/phase1_head/weights/best.pt')
    model.train(
        data=OUTPUT_DIR, epochs=EPOCHS-10, imgsz=IMGSZ, batch=BATCH,
        lr0=LR, lrf=0.01, patience=PATIENCE, device=DEVICE,
        project='training', name='phase2_finetune', exist_ok=True,
        optimizer='AdamW', warmup_epochs=3, cos_lr=True, seed=SEED, workers=2,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=15.0, translate=0.1, scale=0.5, shear=5.0, perspective=5e-4,
        flipud=0.1, fliplr=0.5,
        mosaic=0.0,    # disabled for classification
        mixup=0.1,     # label smoothing via mixup
        erasing=0.3,   # occlusion robustness
        crop_fraction=0.8,
        verbose=True,
    )
    BEST_MODEL = 'training/phase2_finetune/weights/best.pt'
else:
    # Single-phase (matches train_single_phase)
    print("=" * 60)
    print(f"Single-phase training ({EPOCHS} epochs)")
    print("=" * 60)
    model = YOLO(MODEL_NAME)
    model.train(
        data=OUTPUT_DIR, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        lr0=LR, lrf=0.01, patience=PATIENCE, device=DEVICE,
        project='training', name='single_phase', exist_ok=True,
        pretrained=True, optimizer='AdamW', weight_decay=5e-4,
        warmup_epochs=5, warmup_momentum=0.8, cos_lr=True, seed=SEED, workers=2,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=15.0, translate=0.1, scale=0.5, shear=5.0, perspective=5e-4,
        flipud=0.1, fliplr=0.5,
        mosaic=0.0, mixup=0.1, erasing=0.3, crop_fraction=0.8,
        verbose=True,
    )
    BEST_MODEL = 'training/single_phase/weights/best.pt'

print(f"\nBest model: {BEST_MODEL}")
print(f"Size: {os.path.getsize(BEST_MODEL)/1e6:.1f} MB")

---
## 5. Task 2 — Comprehensive Evaluation

Mirrors `task2_model_evaluation.py` — rigorous evaluation producing:
- Per-class precision / recall / F1-score
- **Macro F1** (preferred for imbalanced datasets — weights every class equally, exposing minority-class weakness)
- Confusion matrix (raw + normalised)
- Per-class F1 bar chart
- Confidence calibration (correct vs incorrect distributions)
- **Misclassification audit**: top confusion pairs, high-confidence errors (most dangerous in production), low-confidence correct predictions (most fragile)
- All metrics persisted as JSON

In [ ]:
# YOLO built-in test evaluation
model = YOLO(BEST_MODEL)
val_results = model.val(split='test')
print(f"YOLO Top-1: {val_results.results_dict.get('metrics/accuracy_top1', 0):.4f}")
print(f"YOLO Top-5: {val_results.results_dict.get('metrics/accuracy_top5', 0):.4f}")

In [ ]:
# Per-image inference (matches task2_model_evaluation.run_inference)
test_dir = Path(OUTPUT_DIR) / 'test'
class_names = sorted([d.name for d in test_dir.iterdir() if d.is_dir()])
y_true, y_pred, confidences, predictions = [], [], [], []

for class_dir in sorted(test_dir.iterdir()):
    if not class_dir.is_dir(): continue
    true_label = class_dir.name
    for img_path in sorted(class_dir.iterdir()):
        if img_path.suffix.lower() not in IMG_EXTENSIONS: continue
        preds = model.predict(str(img_path), imgsz=IMGSZ, verbose=False)
        p = preds[0]
        pred_label = p.names[p.probs.top1]
        conf = float(p.probs.top1conf)
        top5_labels = [p.names[i] for i in p.probs.top5]
        top5_confs  = [float(p.probs.data[i]) for i in p.probs.top5]

        y_true.append(true_label)
        y_pred.append(pred_label)
        confidences.append(conf)
        predictions.append({
            'path': str(img_path), 'true_label': true_label,
            'pred_label': pred_label, 'confidence': conf,
            'correct': true_label == pred_label,
            'in_top5': true_label in top5_labels,
            'top5_labels': top5_labels, 'top5_confs': top5_confs,
        })

print(f"Evaluated {len(predictions)} test images")

In [ ]:
# Classification report (matches task2_model_evaluation.print_metrics)
print(classification_report(y_true, y_pred, labels=class_names, zero_division=0))

macro_f1    = f1_score(y_true, y_pred, labels=class_names, average='macro', zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, labels=class_names, average='weighted', zero_division=0)
top1_acc    = sum(p['correct'] for p in predictions) / len(predictions)
top5_acc    = sum(p['in_top5'] for p in predictions) / len(predictions)
correct_c   = [p['confidence'] for p in predictions if p['correct']]
wrong_c     = [p['confidence'] for p in predictions if not p['correct']]

print(f"Top-1 Accuracy:          {top1_acc:.4f}")
print(f"Top-5 Accuracy:          {top5_acc:.4f}")
print(f"Macro F1 (primary):      {macro_f1:.4f}")
print(f"Weighted F1:             {weighted_f1:.4f}")
print(f"Mean conf (correct):     {np.mean(correct_c):.4f}")
print(f"Mean conf (incorrect):   {np.mean(wrong_c):.4f}" if wrong_c else "No errors!")

In [ ]:
# Confusion matrix — raw + normalised (matches task2_model_evaluation.plot_confusion_matrix)
cm = confusion_matrix(y_true, y_pred, labels=class_names)
cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax, data, fmt, title in [
    (axes[0], cm, 'd', 'Confusion Matrix (Counts)'),
    (axes[1], cm_norm, '.2f', 'Confusion Matrix (Normalised)'),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class F1 bar chart (matches task2_model_evaluation.plot_per_class_f1)
report = classification_report(y_true, y_pred, labels=class_names,
                               output_dict=True, zero_division=0)
f1_scores = [report[c]['f1-score'] for c in class_names]
colours = ['#d32f2f' if s < 0.5 else '#ff9800' if s < 0.7 else '#388e3c' for s in f1_scores]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(class_names, f1_scores, color=colours, edgecolor='grey')
for bar, val in zip(bars, f1_scores):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)
ax.axvline(macro_f1, color='blue', ls='--', alpha=0.5, label=f'Macro F1 = {macro_f1:.3f}')
ax.set_xlim(0, 1.15)
ax.set_xlabel('F1 Score')
ax.set_title('Per-Class F1 Scores')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Confidence distribution (matches task2_model_evaluation.plot_confidence_histogram)
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(0, 1, 30)
ax.hist(correct_c, bins=bins, alpha=0.7, color='green',
        edgecolor='darkgreen', label=f'Correct (n={len(correct_c)})')
if wrong_c:
    ax.hist(wrong_c, bins=bins, alpha=0.7, color='red',
            edgecolor='darkred', label=f'Incorrect (n={len(wrong_c)})')
ax.set_xlabel('Confidence')
ax.set_ylabel('Count')
ax.set_title('Prediction Confidence Distribution')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Misclassification audit (matches task2_model_evaluation.misclassification_audit)
errors = [p for p in predictions if not p['correct']]
print("=" * 65)
print("  MISCLASSIFICATION AUDIT")
print("=" * 65)
print(f"Total errors: {len(errors)} / {len(predictions)} ({len(errors)/len(predictions)*100:.1f}%)\n")

if errors:
    # Top confusion pairs
    pair_counts = Counter((p['true_label'], p['pred_label']) for p in errors)
    print("Top confusion pairs (true -> predicted):")
    for (true, pred), cnt in pair_counts.most_common(10):
        print(f"  {true:25s} -> {pred:25s} : {cnt}x")

    # High-confidence errors — most dangerous in production
    print("\nHighest-confidence errors (most dangerous in production):")
    for p in sorted(errors, key=lambda x: -x['confidence'])[:5]:
        print(f"  {p['true_label']:25s} predicted as {p['pred_label']:25s}"
              f"  conf={p['confidence']:.3f}  ({Path(p['path']).name})")

    # Low-confidence correct — most fragile
    fragile = sorted([p for p in predictions if p['correct']], key=lambda x: x['confidence'])[:5]
    print("\nLowest-confidence correct (fragile — likely to flip with noise):")
    for p in fragile:
        print(f"  {p['true_label']:25s}  conf={p['confidence']:.3f}")
else:
    print("Perfect score — no misclassifications!")
print("=" * 65)

In [ ]:
# Save metrics + predictions as JSON (matches task2_model_evaluation persistence)
metrics_out = {
    'top1_accuracy': top1_acc, 'top5_accuracy': top5_acc,
    'macro_f1': macro_f1, 'weighted_f1': weighted_f1,
    'total_samples': len(predictions),
    'correct': sum(p['correct'] for p in predictions),
    'errors': len(errors),
    'mean_conf_correct': float(np.mean(correct_c)) if correct_c else 0,
    'mean_conf_wrong': float(np.mean(wrong_c)) if wrong_c else 0,
    'per_class': report,
    'confusion_matrix': cm.tolist(),
    'class_names': class_names,
}

with open('evaluation_metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2, default=str)
with open('evaluation_predictions.json', 'w') as f:
    json.dump(predictions, f, indent=2)

print("Saved evaluation_metrics.json")
print("Saved evaluation_predictions.json")

---
## 6. Export Model to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_EXPORT_DIR = '/content/drive/MyDrive/rtv_classifier'  # Edit if needed
os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)

# Copy model weights
shutil.copy2(BEST_MODEL, os.path.join(DRIVE_EXPORT_DIR, 'best.pt'))
print(f"Model:   {DRIVE_EXPORT_DIR}/best.pt ({os.path.getsize(BEST_MODEL)/1e6:.1f} MB)")

last = BEST_MODEL.replace('best.pt', 'last.pt')
if os.path.exists(last):
    shutil.copy2(last, os.path.join(DRIVE_EXPORT_DIR, 'last.pt'))
    print(f"Backup:  {DRIVE_EXPORT_DIR}/last.pt")

# Copy evaluation files
for f in ['evaluation_metrics.json', 'evaluation_predictions.json']:
    if os.path.exists(f):
        shutil.copy2(f, os.path.join(DRIVE_EXPORT_DIR, f))

print(f"\nAll exports in {DRIVE_EXPORT_DIR}:")
for f in sorted(os.listdir(DRIVE_EXPORT_DIR)):
    print(f"  {f}: {os.path.getsize(os.path.join(DRIVE_EXPORT_DIR, f))/1e6:.1f} MB")

In [ ]:
# Or download directly to your machine
from google.colab import files
files.download(BEST_MODEL)

---
## 7. Next Steps — Local Deployment

After downloading `best.pt`:

```bash
# Place the model
mkdir -p outputs/training
cp ~/Downloads/best.pt outputs/training/best.pt

# Run FastAPI server
export MODEL_PATH=outputs/training/best.pt
uvicorn src.app.main:app --host 0.0.0.0 --port 8000

# Or deploy with Docker
docker compose up --build

# Open http://localhost:8000 to test with the interactive UI
# Open http://localhost:8000/docs for Swagger API docs
```